<a href="https://colab.research.google.com/github/vabnamadina/Image-Preprocessing/blob/main/Image_pre_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: mount Google Drive (this was the missing step -> FileNotFoundError)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: AVIF support for PIL (needed for .avif files)
!pip install -q pillow-avif-plugin

In [ ]:
# ============================================================
# SKIN IMAGE PREPROCESSING PIPELINE - Normal Skin Dataset
# Fix brightness + contrast instead of rejecting
# ============================================================
import os
import cv2
import numpy as np
from PIL import Image

try:
    import pillow_avif  # registers AVIF decoder with PIL
except ImportError:
    print("pillow-avif-plugin not installed: .avif files will fail to read")

# ------------------------------------------------------------
# PATH CONFIGURATION
# ------------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive"
RAW_DIR    = os.path.join(DRIVE_ROOT, "Colab Notebooks", "Dataset", "raw_normal_skin")
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "Colab Notebooks", "Dataset", "preprocessed_normal_skin")

IMG_SIZE = (224, 224)
CLEAN_OUTPUT = False   # True = delete old .jpg files in OUTPUT_DIR before running
VALID_EXT = (".jpg", ".jpeg", ".png", ".webp", ".avif")


def find_folder(name, root=DRIVE_ROOT):
    """Search Drive for a folder with this name (used if the path above is wrong)."""
    for dirpath, dirnames, _ in os.walk(root):
        if name in dirnames:
            return os.path.join(dirpath, name)
    return None


# If the hard-coded path is wrong, locate the folder automatically
if not os.path.isdir(RAW_DIR):
    print("Path not found:", RAW_DIR)
    found = find_folder("raw_normal_skin")
    if found is None:
        raise FileNotFoundError(
            "Could not find 'raw_normal_skin' anywhere in Google Drive.\n"
            "Check the folder name / that the Drive is mounted, then edit RAW_DIR."
        )
    RAW_DIR = found
    OUTPUT_DIR = os.path.join(os.path.dirname(found), "preprocessed_normal_skin")
    print("Found it automatically:", RAW_DIR)

print("RAW_DIR   :", RAW_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


# ------------------------------------------------------------
# IMAGE READING
# ------------------------------------------------------------
def read_image(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".webp", ".avif"):
        img = np.array(Image.open(path).convert("RGB"))
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    else:
        # np.fromfile + imdecode also works with unicode / odd paths
        data = np.fromfile(path, dtype=np.uint8)
        img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("Cannot read image")
    return img


# ------------------------------------------------------------
# RESIZE WITH PADDING
# ------------------------------------------------------------
def resize_with_padding(image, size=(224, 224)):
    h, w = image.shape[:2]
    target_w, target_h = size
    scale = min(target_w / w, target_h / h)
    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    x = (target_w - new_w) // 2
    y = (target_h - new_h) // 2
    canvas[y:y + new_h, x:x + new_w] = resized
    return canvas


# ------------------------------------------------------------
# BRIGHTNESS CORRECTION (gamma)
# ------------------------------------------------------------
def adjust_brightness(image, target_mean=130):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    current_mean = float(np.mean(gray))
    # keep strictly inside (0, 255) so log() is valid and never divides by 0
    current_mean = np.clip(current_mean, 1, 254)

    gamma = np.log(target_mean / 255.0) / np.log(current_mean / 255.0)
    gamma = np.clip(gamma, 0.5, 2.0)

    lut = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(image, lut)


# ------------------------------------------------------------
# CONTRAST ENHANCEMENT (CLAHE on L channel)
# ------------------------------------------------------------
def enhance_contrast(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l2 = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l2, a, b)), cv2.COLOR_LAB2BGR)


# ------------------------------------------------------------
# DENOISING
# ------------------------------------------------------------
def denoise_image(image):
    return cv2.fastNlMeansDenoisingColored(image, None, 5, 5, 7, 21)


# ------------------------------------------------------------
# BLUR CHECK (only reject extremely blurry images)
# Done on the ORIGINAL image: denoising smooths the image and
# CLAHE amplifies edges, so checking afterwards gives misleading scores.
# ------------------------------------------------------------
def blur_check(image, threshold=5):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur_value = cv2.Laplacian(gray, cv2.CV_64F).var()
    return blur_value >= threshold, blur_value


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
def preprocessing_normal_skin(raw_dir, output_dir):
    if not os.path.isdir(raw_dir):
        raise FileNotFoundError(f"Dataset folder not found: {raw_dir}")

    os.makedirs(output_dir, exist_ok=True)

    if CLEAN_OUTPUT:  # only removes old .jpg outputs, never the folder itself
        for f in os.listdir(output_dir):
            if f.lower().endswith(".jpg"):
                os.remove(os.path.join(output_dir, f))

    images = sorted(f for f in os.listdir(raw_dir) if f.lower().endswith(VALID_EXT))
    print("Total images found:", len(images))
    if not images:
        print("No supported images in:", raw_dir)
        print("Folder contains:", os.listdir(raw_dir)[:20])
        return

    accepted = rejected = 0
    used_names = set()

    for img_name in images:
        img_path = os.path.join(raw_dir, img_name)
        try:
            img = read_image(img_path)

            ok_blur, blur_score = blur_check(img, threshold=5)
            if not ok_blur:
                rejected += 1
                print(f"Rejected (blurry, score={blur_score:.2f}): {img_name}")
                continue

            img = adjust_brightness(img, target_mean=130)
            img = enhance_contrast(img)
            img = denoise_image(img)
            img = resize_with_padding(img, IMG_SIZE)

            # avoid overwriting when e.g. a.png and a.jpg both exist
            stem = os.path.splitext(img_name)[0]
            save_name = stem + ".jpg"
            if save_name in used_names:
                save_name = f"{stem}_{os.path.splitext(img_name)[1][1:]}.jpg"
            used_names.add(save_name)

            ok = cv2.imwrite(os.path.join(output_dir, save_name), img,
                             [cv2.IMWRITE_JPEG_QUALITY, 95])
            if not ok:
                raise IOError("cv2.imwrite failed")
            accepted += 1

        except Exception as e:
            print("Error:", img_name, "->", e)
            rejected += 1

    print("\n====================")
    print("Preprocessing Completed")
    print("====================")
    print("Accepted images:", accepted)
    print("Rejected images:", rejected)
    print("Saved at:", output_dir)


preprocessing_normal_skin(RAW_DIR, OUTPUT_DIR)